# hidden-state 덤프 (exp1 8192 · 고정 풀링)

exp1(`ingyoun/A.X-patent-maxlen8192`)의 마지막 은닉 상태를 한 번의 forward로 잡아 **네 가지 고정 풀링**(mean·max·cls·last)으로 문서 벡터 `[N,768]`를 만들어 Drive에 덤프한다.

- **목적**: 장문 열화가 mean 풀링의 신호 희석 때문인지(→ max 등 다른 풀링으로 회수) 본질적 난이도인지를 오프라인(로컬 CPU)에서 가르는 프로브의 입력 특징을 만든다.
- `mean`은 모델이 실제로 쓰는 풀링(`classifier_pooling='mean'`)과 동일해, 같은 forward의 logits로 복원 정합을 검증한다(정리 test SSOT micro **0.8683** — ADR-0010 `headline_cleaned_test.json`).
- 데이터: 정리본(11,244/11,132)을 쓴다.
- `patent_train`은 수정하지 않는다 — 추론 러너를 노트북에서 상속해 `predict_hidden`만 얹는다.

In [ ]:
# flash-attn 프리빌트 휠(cu12·torch2.11·cp312) — 백본이 flash_attention_2를 요구
!wget -q "https://github.com/lesj0610/flash-attention/releases/download/v2.8.3-cu12-torch2.11/flash_attn-2.8.3+cu12torch2.11cxx11abiTRUE-cp312-cp312-linux_x86_64.whl"
!pip install -q flash_attn-2.8.3+cu12torch2.11cxx11abiTRUE-cp312-cp312-linux_x86_64.whl
# transformers 버전 pin — 로컬 .venv(5.13.0)와 같은 ModernBERT+FA2 regime 고정(colab-jobs.md)
!pip install -q "transformers==5.13.0" "accelerate>=1.1.0"

In [ ]:
from google.colab import drive

drive.mount("/content/drive")

In [ ]:
import os, sys, shutil

DRIVE = "/content/drive/MyDrive/patent_disc"        # Drive 프로젝트 루트(코드·산출물)

# 코드 반입 — Drive의 patent_train을 로컬 /content/src로 복사해 import(네트워크 I/O 회피).
shutil.copytree(f"{DRIVE}/src/patent_train", "/content/src/patent_train",
                dirs_exist_ok=True, ignore=shutil.ignore_patterns("__pycache__"))
sys.path.insert(0, "/content/src")

import numpy as np
import torch
import transformers
import patent_train
from patent_train import TrainConfig, TrainingRunner

print("patent_train :", patent_train.__file__)   # /content/src/patent_train/__init__.py 여야
print("transformers :", transformers.__version__)
print("torch        :", torch.__version__, "| cuda:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "CPU")

In [ ]:
from google.colab import userdata

os.environ["HF_TOKEN"] = userdata.get("HUGGINGFACEHUB_API_TOKEN")   # Hub 모델·데이터셋 다운로드
os.environ["HF_HOME"] = "/content/.hf_cache"                        # 휘발 — 매 VM 새로 받는다
os.environ["HF_HUB_DISABLE_XET"] = "1"

## 상속 — hidden-state 덤프 러너

`TrainingRunner`를 상속해 `predict_hidden`을 추가한다. `model.model`(ModernBertModel 백본)에 forward hook을 걸어 마지막 은닉 상태 `h[B,T,768]`를, 같은 forward의 `out.logits`(실제 모델 로짓)로 복원을 검증한다. 풀링 4종을 `h`에서 계산한다

In [ ]:
class HiddenDumpRunner(TrainingRunner):
    """추론 러너를 상속해 hidden-state 풀링 벡터를 덤프한다(patent_train 불변)."""

    POOLINGS = ("mean", "max", "cls", "last")

    @torch.no_grad()
    def predict_hidden(self, split: str, out_dir: str):
        """split의 마지막 은닉 상태를 4종 풀링해 `hidden_{tag}_{split}_{pool}.npy`로 덤프한다.

        한 번의 forward로 (a) 훅으로 백본 출력 h를 잡아 mean/max/cls/last 풀링,
        (b) out.logits로 micro를 재 복원 정합을 검증한다. `predict_logits`와 같이
        덤프 동안 샘플러를 순차로 되돌리고 행 순서를 assert로 못박는다.
        """
        self._require_trainer()
        ds = self.data.dataset[split]
        model = self.trainer.model.eval()
        device = model.device

        cap = {}
        def _hook(_m, _in, out):
            cap["h"] = out.last_hidden_state if hasattr(out, "last_hidden_state") else out[0]
        handle = model.model.register_forward_hook(_hook)   # ModernBertModel(백본) 출력 포착

        pools = {p: [] for p in self.POOLINGS}
        logits_all, labels_all = [], []

        args = self.trainer.args
        strat, args.train_sampling_strategy = args.train_sampling_strategy, "sequential"  # 행 순서 보장
        try:
            for batch in self.trainer.get_eval_dataloader(ds):
                # get_eval_dataloader는 accelerate가 배치를 GPU로 올린다 — 라벨도 cuda라 cpu()로 모은다.
                labels = batch["labels"].cpu()
                ids = batch["input_ids"].to(device)
                mask = batch["attention_mask"].to(device)
                with torch.autocast("cuda", dtype=torch.bfloat16):
                    out = model(input_ids=ids, attention_mask=mask)   # 실제 logits + 훅으로 h
                h = cap["h"].float()                                  # [B,T,768]
                m = mask.unsqueeze(-1).float()                        # [B,T,1]
                pools["mean"].append(((h * m).sum(1) / m.sum(1).clamp(min=1)).half().cpu())
                pools["max"].append(h.masked_fill(m == 0, float("-inf")).max(1).values.half().cpu())
                pools["cls"].append(h[:, 0].half().cpu())
                idx = mask.sum(1).long() - 1                          # 마지막 실토큰(eos, 우측 패딩)
                pools["last"].append(h[torch.arange(h.size(0), device=device), idx].half().cpu())
                logits_all.append(out.logits.float().cpu())
                labels_all.append(labels)
        finally:
            args.train_sampling_strategy = strat
            handle.remove()

        pooled = {p: torch.cat(v).numpy() for p, v in pools.items()}
        logits = torch.cat(logits_all).numpy()
        labels = torch.cat(labels_all).numpy()
        # 행 순서 보증 — 수집 라벨이 데이터셋 라벨과 행 단위로 같아야 한다(순열 섞임 차단).
        assert np.array_equal(labels, np.asarray(ds["labels"], dtype=np.float32)), \
            f"{split} 행 순서가 데이터셋과 다르다 — eval 샘플러 확인"
        # 같은 forward의 logits로 복원 정합 검증(mean 풀링 == 모델 내부 풀링).
        self.metrics[split] = self.trainer.compute_metrics((logits, labels))

        os.makedirs(out_dir, exist_ok=True)
        for p, arr in pooled.items():
            fp = os.path.join(out_dir, f"hidden_{self.cfg.tag}_{split}_{p}.npy")
            np.save(fp, arr)
            print(f"[dump] {fp}  shape={arr.shape} dtype={arr.dtype}")
        return pooled

In [ ]:
cfg = TrainConfig.for_inference(
    tag="modernbert-patent-len8192",                # 로짓 덤프와 같은 tag(파일명 접두어)
    checkpoint="ingyoun/A.X-patent-maxlen8192",     # exp1(8192) — 04_02가 push
    out_path="/content/output/hidden",              # 미사용(덤프는 아래 out_dir로 Drive 직결)
    workspace="/content",
    max_len=8192,                                   # exp1 full length
    eval_micro_batch=8,                             # 8192 추론 배치(L4 24GB, KD 공통 프로토콜)
    splits=("val", "test"),                         # train(201k행) 로드 회피
)
print("checkpoint:", cfg.checkpoint, "| splits:", cfg.splits, "| max_len:", cfg.max_len)

## 로드 — 모델·데이터셋(Hub)

`load_data`(토크나이저 + 원본 val/test) → `prepare_data`(max_len 절단) → `load_model`(exp1 복원) → `build_trainer`(추론 구성: wandb·early stop·save 없음, 순차 샘플러)

In [ ]:
runner = HiddenDumpRunner(cfg)
runner.load_data()       # 토크나이저 + 원본(val/test) 로드
runner.prepare_data()    # max_len=8192 절단
runner.load_model()      # exp1 체크포인트 복원
runner.build_trainer()   # 추론 전용 Trainer
runner.data.dataset

## 덤프 · 검증 (hidden → Drive)

split마다 `predict_hidden`으로 4종 풀링 벡터를 Drive에 저장하고, 같은 forward의 logits로 잰 test micro가 정리 test SSOT(0.8683)와 맞는지로 체크포인트 정상 복원·행 순서를 확인한다.

In [ ]:
SSOT_TEST_MICRO = 0.8683              # ADR-0010 headline_cleaned_test.json (exp1 정리 test)
HIDDEN_DIR = f"{DRIVE}/output"        # 로짓과 같은 디렉터리에 hidden_{tag}_{split}_{pool}.npy

for split in ("val", "test"):
    runner.predict_hidden(split, out_dir=HIDDEN_DIR)
    m = runner.metrics[split]
    print(f"[{split}] micro_f1 {m['micro_f1']:.4f} · macro {m['macro_f1']:.4f} · sample {m['sample_f1']:.4f}")

assert abs(runner.metrics["test"]["micro_f1"] - SSOT_TEST_MICRO) < 2e-3, runner.metrics["test"]
print(f"verify: test micro ≈ SSOT {SSOT_TEST_MICRO} — exp1 정상 복원 · 행 순서 정합 · mean 풀링==모델 풀링")